# DistilBERT para triagem de OS (Fase 3)

Este notebook só **chama o pacote `mp`** (a lógica está em `src/mp`, testada). Ele treina o
`distilbert-base-multilingual-cased` na GPU, exporta ONNX INT8 e devolve 3 arquivos para o repositório.

**Antes de rodar:** `Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4`.
Você precisa do arquivo `mp-colab.zip` (gerado por `python scripts/empacotar_colab.py`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Sem GPU: troque o tipo de ambiente de execução para GPU T4"

## 1. Enviar o pacote e (opcional) guardar o checkpoint no Drive

In [ ]:
from google.colab import files
import zipfile, os
enviado = files.upload()                     # selecione mp-colab.zip
zipfile.ZipFile(next(iter(enviado))).extractall("/content/mp")
os.chdir("/content/mp")
print(sorted(os.listdir(".")))

In [ ]:
# Opcional, mas recomendado: a sessão do Colab pode cair. Com isto o melhor checkpoint
# (artifacts/bert/melhor.pt) fica no seu Drive a cada época.
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/mp_bert", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)
if not os.path.exists("artifacts/bert"):
    os.symlink("/content/drive/MyDrive/mp_bert", "artifacts/bert")

## 2. Instalar o pacote

In [ ]:
!pip -q install -e ".[bert]"

## 3. Treinar

Mesma divisão por motor e mesma seed do baseline TF-IDF, para a comparação ser justa.
Na T4, 4 épocas levam alguns minutos (fp16). O modelo (~540 MB) é baixado do Hugging Face aqui dentro.

In [ ]:
!python -m mp.train_bert --epocas 4 --batch 32 --max-len 64

## 4. Conferir o resultado

In [ ]:
import json
m = json.load(open("artifacts/bert/metrics.json", encoding="utf-8"))
for nome in ("distilbert_fp32", "distilbert_int8"):
    r = m[nome]
    print(nome, "| F1-macro ruidoso", round(r["ruidoso"]["f1_macro"], 3),
          "| referência", round(r["referencia"]["f1_macro"], 3),
          "| recall crítica ruidoso", round(r["ruidoso"]["recall"]["crítica"], 3),
          "| referência", round(r["referencia"]["recall"]["crítica"], 3))
print("paridade:", m["paridade"])
print("tamanhos (MB):", m["tamanho_mb"], "| treino (s):", m["treino_s"])

## 5. Levar os arquivos de volta

Baixe os 3 arquivos abaixo e coloque-os em `artifacts/bert/` no repositório. Depois, no seu computador:
`python -m mp.exportar_servico` e reinicie a API (`podman compose restart api`).

In [ ]:
import shutil
os.makedirs("/content/saida", exist_ok=True)
for f in ("bert_int8.onnx", "tokenizer.json", "metrics.json"):
    shutil.copy(f"artifacts/bert/{f}", "/content/saida/")
shutil.make_archive("/content/bert-saida", "zip", "/content/saida")
files.download("/content/bert-saida.zip")